# 02 - Stockouts and unconstrained-demand reconstruction

This notebook joins the cleaned history to `inventory_stockouts.csv`. It keeps three concepts separate:
1. `Units_Sold` - the cleaned value from the sales extract.
2. `Units_Sold_Observed` - observed shipped units, using stockout `Shipped_Units` when the stockout table provides direct evidence.
3. `Unconstrained_Demand` - `Requested_Units` for listed stockout months; otherwise observed shipments as a proxy.

`Requested_Units` is current-month demand. Do **not** add `Carried_In_Units` to it: carried-in units were requested in a previous month. The stockout file is incomplete, so an unlisted month is not proof that supply was unconstrained. Missing sales rows are not converted to zero.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "outputs" / "cleaned").is_dir():
            return candidate
    raise FileNotFoundError("Run notebook 01 first; outputs/cleaned was not found.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.demand_reconstruction import run_demand_reconstruction

summary = run_demand_reconstruction(PROJECT_ROOT)
for key, value in summary.items():
    if key not in {"outputs", "assumptions"}:
        print(f"{key}: {value}")
print("\nOutputs:")
for name, path in summary["outputs"].items():
    print(f"{name}: {path}")

In [ ]:
import pandas as pd
from IPython.display import display

quality_report = pd.read_csv(
    PROJECT_ROOT / "outputs" / "demand_reconstruction" / "stockout_quality_report.csv"
)
display(quality_report)

panel = pd.read_csv(
    PROJECT_ROOT / "outputs" / "demand_reconstruction" / "train_with_reconstructed_demand.csv",
    low_memory=False,
)
display(panel[[
    "Date", "SKU_ID", "Region", "Units_Sold", "Units_Sold_Observed",
    "Requested_Units", "Unconstrained_Demand", "Demand_Estimate_Source",
    "Demand_Confidence", "Stockout_Row_Added",
]].head())

complete = panel.dropna(subset=["Date", "SKU_ID", "Region"])
assert not complete.duplicated(["Date", "SKU_ID", "Region"]).any()
assert int(panel["Stockout_Row_Added"].fillna(False).sum()) == summary["stockout_records_without_train_row"]
assert summary["stockout_records_matched_to_train"] + summary["stockout_records_without_train_row"] == summary["stockout_records"]
print("Validation passed: stockout keys accounted for and complete panel keys unique.")

## Review before feature engineering
Inspect `stockout_join_audit.csv` for sales/shipment disagreements and `stockout_quality_report.csv` for arithmetic or coverage issues. A request value is a direct demand observation; a shipment-only proxy has lower confidence because missing stockout records may hide other constrained months. Do not treat carried-in units as new-month demand.